# 2022 FIFA World Cup Final — Veri Boru Hattı
**Arjantin vs Fransa** | Veri Çekimi · Önişleme · Kayıt

Bu notebook veriyi StatsBomb'dan çeker, temizler ve `cache/` klasörüne kaydeder.
Görselleştirmeler için `02_visualizations.ipynb` dosyasını çalıştırın.

## 0. Kurulum

In [12]:
!pip install statsbombpy pandas numpy -q

## 1. Kütüphaneler

In [13]:
import os, io, time, warnings
import numpy as np
import pandas as pd
from statsbombpy import sb
warnings.filterwarnings('ignore')

CACHE_DIR = r'D:\Masaüstü\Projects\Match_Analysis2\cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache klasoru: {CACHE_DIR}')

Cache klasoru: D:\Masaüstü\Projects\Match_Analysis2\cache


## 2. Veri Çekimi

In [14]:
def _fetch_with_retry(fn, *args, **kwargs):
    """Rate-limit hatalarinda otomatik yeniden dener."""
    for attempt in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f'  Hata: {e}  |  {wait}s bekleniyor... ({attempt+1}/6)')
            time.sleep(wait)
    raise RuntimeError('API 6 denemede de erisemedi.')

# Yarismalar
cache_comps = os.path.join(CACHE_DIR, 'competitions.json')
if os.path.exists(cache_comps):
    comps = pd.read_json(cache_comps)
    print('[CACHE] competitions yuklendi.')
else:
    comps = _fetch_with_retry(sb.competitions)
    comps.to_json(cache_comps)
    print('[API]   competitions cekildi ve kaydedildi.')

print(comps[comps['competition_name'] == 'FIFA World Cup'][['competition_id','season_id','season_name']].to_string())

[CACHE] competitions yuklendi.
    competition_id  season_id season_name
30              43        106        2022
31              43          3        2018
32              43         55        1990
33              43         54        1986
34              43         51        1974
35              43        272        1970
36              43        270        1962
37              43        269        1958


In [15]:
# 2022 Dunya Kupasi
wc = comps[comps['competition_name'] == 'FIFA World Cup']
row_2022 = wc[wc['season_name'] == '2022'].iloc[0]
competition_id = int(row_2022['competition_id'])
season_id      = int(row_2022['season_id'])
print(f'Competition ID: {competition_id}  |  Season ID: {season_id}')

# Maclar
cache_matches = os.path.join(CACHE_DIR, f'matches_{competition_id}_{season_id}.json')
if os.path.exists(cache_matches):
    matches = pd.read_json(cache_matches)
    print('[CACHE] matches yuklendi.')
else:
    matches = _fetch_with_retry(sb.matches, competition_id=competition_id, season_id=season_id)
    matches.to_json(cache_matches)
    print('[API]   matches cekildi ve kaydedildi.')

print(f'Toplam mac sayisi: {len(matches)}')

Competition ID: 43  |  Season ID: 106
[CACHE] matches yuklendi.
Toplam mac sayisi: 64


In [16]:
# Arjantin - Fransa finalini bul
mask = (
    matches['home_team'].str.contains('Argentina', case=False, na=False) |
    matches['away_team'].str.contains('Argentina', case=False, na=False)
) & (
    matches['home_team'].str.contains('France', case=False, na=False) |
    matches['away_team'].str.contains('France', case=False, na=False)
)
final_row = matches[mask].iloc[0]
match_id  = int(final_row['match_id'])

print(f'Mac ID  : {match_id}')
print(f'Sonuc   : {final_row["home_team"]}  {final_row["home_score"]} - {final_row["away_score"]}  {final_row["away_team"]}')
print(f'Tarih   : {final_row["match_date"]}')

Mac ID  : 3869685
Sonuc   : Argentina  3 - 3  France
Tarih   : 2022-12-18


In [17]:
# Eventleri cek
cache_events = os.path.join(CACHE_DIR, f'events_{match_id}.json')
if os.path.exists(cache_events):
    events_raw = pd.read_json(cache_events)
    print('[CACHE] events yuklendi.')
else:
    events_raw = _fetch_with_retry(sb.events, match_id=match_id)
    events_raw.to_json(cache_events)
    print('[API]   events cekildi ve kaydedildi.')

print(f'Toplam event : {len(events_raw)}')
print(f'Sutun sayisi : {len(events_raw.columns)}')
print(events_raw['type'].value_counts().to_string())

[CACHE] events yuklendi.
Toplam event : 4407
Sutun sayisi : 94
type
Pass               1263
Ball Receipt*      1114
Carry               940
Pressure            361
Ball Recovery       115
Duel                 98
Dribble              54
Block                50
Foul Committed       48
Clearance            45
Foul Won             44
Goal Keeper          44
Shot                 38
Miscontrol           35
Dispossessed         34
Dribbled Past        31
Interception         28
Substitution         13
Half Start           10
Half End             10
Injury Stoppage       9
50/50                 8
Tactical Shift        7
Starting XI           2
Bad Behaviour         2
Player Off            1
Offside               1
Player On             1
Shield                1


## 3. Veri Ön İşleme

In [18]:
events = events_raw.copy()

# location -> x, y
events['x'] = events['location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
events['y'] = events['location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# pas bitis konumu
if 'pass_end_location' in events.columns:
    events['pass_end_x'] = events['pass_end_location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
    events['pass_end_y'] = events['pass_end_location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# tasima bitis konumu
if 'carry_end_location' in events.columns:
    events['carry_end_x'] = events['carry_end_location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
    events['carry_end_y'] = events['carry_end_location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# Takim isimleri
teams = events['team'].dropna().unique().tolist()
ARG   = next(t for t in teams if 'Argentina' in t)
FRA   = next(t for t in teams if 'France'    in t)

print(f'Takimlar: "{ARG}"  vs  "{FRA}"')
print(f'x sutunu bos olmayan event sayisi: {events["x"].notna().sum()}')

Takimlar: "Argentina"  vs  "France"
x sutunu bos olmayan event sayisi: 4352


In [19]:
# Eksik veri ozeti
null_pct = (events.isnull().sum() / len(events) * 100).round(1)
print('=== EKSIK VERI OZETI (ilk 20 sutun) ===')
print(null_pct[null_pct > 0].sort_values(ascending=False).head(20).to_string())

=== EKSIK VERI OZETI (ilk 20 sutun) ===
bad_behaviour_card          100.0
ball_recovery_offensive     100.0
block_deflection            100.0
foul_committed_advantage    100.0
clearance_other             100.0
block_offensive             100.0
pass_deflected              100.0
foul_won_penalty            100.0
shot_aerial_won             100.0
pass_goal_assist            100.0
foul_won_advantage          100.0
foul_committed_offensive    100.0
dribble_overrun              99.9
dribble_nutmeg               99.9
foul_committed_penalty       99.9
foul_committed_type          99.9
pass_inswinging              99.9
shot_one_on_one              99.9
foul_committed_card          99.9
pass_through_ball            99.9


## 4. Maç İstatistikleri

In [20]:
def team_stats(events, team):
    te       = events[events['team'] == team]
    passes   = te[te['type'] == 'Pass']
    shots    = te[te['type'] == 'Shot']
    goals    = shots[shots['shot_outcome'] == 'Goal']
    dribbles = te[te['type'] == 'Dribble']
    presses  = te[te['type'] == 'Pressure']
    return {
        'Pas'           : len(passes),
        'Pas Basari (%)': round(100 * passes['pass_outcome'].isna().sum() / max(len(passes), 1), 1),
        'Sut'           : len(shots),
        'Gol'           : len(goals),
        'Dribling'      : len(dribbles),
        'Baski'         : len(presses),
        'Top. Event'    : len(te),
    }

stats = pd.DataFrame([team_stats(events, ARG), team_stats(events, FRA)], index=[ARG, FRA]).T
print('=== MAC ISTATISTIKLERI ===')
print(stats.to_string())

=== MAC ISTATISTIKLERI ===
                Argentina  France
Pas                 693.0   570.0
Pas Basari (%)       80.8    76.1
Sut                  24.0    14.0
Gol                   7.0     5.0
Dribling             21.0    33.0
Baski               167.0   194.0
Top. Event         2372.0  2035.0


In [21]:
print('=== ARJANTiN OYUNCULARI ===')
print(events[events['team']==ARG]['player'].dropna().unique().tolist())
print('\n=== FRANSA OYUNCULARI ===')
print(events[events['team']==FRA]['player'].dropna().unique().tolist())

=== ARJANTiN OYUNCULARI ===
['Nahuel Molina Lucero', 'Rodrigo Javier De Paul', 'Cristian Gabriel Romero', 'Nicolás Hernán Otamendi', 'Nicolás Alejandro Tagliafico', 'Damián Emiliano Martínez', 'Alexis Mac Allister', 'Lionel Andrés Messi Cuccittini', 'Ángel Fabián Di María Hernández', 'Enzo Fernandez', 'Julián Álvarez', 'Marcos Javier Acuña', 'Gonzalo Ariel Montiel', 'Leandro Daniel Paredes', 'Lautaro Javier Martínez', 'Paulo Bruno Exequiel Dybala', 'Germán Alejandro Pezzella']

=== FRANSA OYUNCULARI ===
['Antoine Griezmann', 'Aurélien Djani Tchouaméni', 'Theo Bernard François Hernández', 'Adrien Rabiot', 'Raphaël Varane', 'Jules Koundé', 'Hugo Lloris', 'Olivier Giroud', 'Kylian Mbappé Lottin', 'Ousmane Dembélé', 'Dayotchanculle Upamecano', 'Randal Kolo Muani', 'Marcus Thuram', 'Eduardo Camavinga', 'Kingsley Coman', 'Youssouf Fofana', 'Ibrahima Konaté']


## 5. İşlenmiş Veriyi Kaydet

In [22]:
# Metadata (takim isimleri, mac id) ayri kaydet
meta = {'ARG': ARG, 'FRA': FRA, 'match_id': match_id,
        'competition_id': competition_id, 'season_id': season_id}

import json
with open(os.path.join(CACHE_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False)

# Isleninis events DataFrame'ini pickle olarak kaydet
pkl_path = os.path.join(CACHE_DIR, 'events_processed.pkl')
events.to_pickle(pkl_path)

print(f'events_processed.pkl kaydedildi  ({os.path.getsize(pkl_path)//1024} KB)')
print(f'meta.json          kaydedildi')
print('\nGorusturmeler icin 02_visualizations.ipynb dosyasini calistirin.')

events_processed.pkl kaydedildi  (3189 KB)
meta.json          kaydedildi

Gorusturmeler icin 02_visualizations.ipynb dosyasini calistirin.
